# 01 · Explore NY workers' comp claims

**Goal:** understand the data before modeling attorney representation (litigation).

Data: NY Workers' Compensation Board (WCB) *Assembled Workers' Compensation Claims*, loaded by `sql/02_load_raw.sql`
and cleaned by the view `WC_ML.ANALYTICS.CLAIM_FEATURES` (`sql/03_features.sql`).

Everything here runs inside Snowflake. Only small summary tables come back to the notebook.

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
import pandas as pd

session = get_active_session()
session.use_warehouse("WC_ML_WH")
pd.set_option("display.max_columns", 50)

def q(sql: str) -> pd.DataFrame:
    """Run SQL in Snowflake and return a pandas DataFrame."""
    return session.sql(sql).to_pandas()

## 1. What did we load?

In [ ]:
q("SELECT COUNT(*) AS N_CLAIMS, MIN(ASSEMBLY_DATE) AS FIRST_ASSEMBLY, MAX(ASSEMBLY_DATE) AS LAST_ASSEMBLY FROM WC_ML.RAW.NY_WCB_CLAIMS")

In [ ]:
# All raw columns (every one is text on purpose; the feature view does the casting)
q("DESCRIBE TABLE WC_ML.RAW.NY_WCB_CLAIMS")[["name", "type"]]

In [ ]:
# The label's raw values: confirm they are Y/N before trusting IS_LITIGATED
q("""
SELECT ATTORNEY_REPRESENTATIVE, COUNT(*) AS N
FROM WC_ML.RAW.NY_WCB_CLAIMS GROUP BY 1 ORDER BY 2 DESC
""")

## 2. Label maturity

An attorney can join a claim months or years after it is assembled. Recent years therefore show a
*lower* litigation rate simply because they have had less time. This is a censoring problem, so:

* train and test on years that are mature enough (default: 2017–2022), and
* split by **time**, not at random, so the test mimics scoring future claims.

In [ ]:
q("""
SELECT ASSEMBLY_YEAR, COUNT(*) AS N_CLAIMS, ROUND(AVG(IS_LITIGATED), 4) AS LITIGATION_RATE
FROM WC_ML.ANALYTICS.CLAIM_FEATURES GROUP BY 1 ORDER BY 1
""")

## 3. Missing values in the model features

In [ ]:
features = session.table("WC_ML.ANALYTICS.CLAIM_FEATURES")
numeric = ["AGE_AT_INJURY", "REPORT_LAG_DAYS", "ACCIDENT_MONTH", "ACCIDENT_DOW"]
categorical = ["GENDER", "CARRIER_TYPE", "DISTRICT_NAME", "COUNTY_OF_INJURY", "MEDICAL_FEE_REGION",
               "INDUSTRY_CODE", "WCIO_CAUSE_CODE", "WCIO_NATURE_CODE", "WCIO_BODY_CODE", "COVID_19_INDICATOR"]

aggs = [F.avg(F.iff(F.col(c).is_null(), 1, 0)).alias(c) for c in numeric] + \
       [F.avg(F.iff(F.col(c) == "UNKNOWN", 1, 0)).alias(c) for c in categorical]
features.agg(*aggs).to_pandas().T.rename(columns={0: "share_missing"}).round(3)

## 4. Which claim characteristics go with litigation?

In [ ]:
def rate_by(column: str, min_n: int = 1000, top: int = 20) -> pd.DataFrame:
    """Litigation rate per category, for categories with at least min_n claims."""
    return q(f"""
        SELECT {column}, COUNT(*) AS N, ROUND(AVG(IS_LITIGATED), 3) AS LITIGATION_RATE
        FROM WC_ML.ANALYTICS.CLAIM_FEATURES
        WHERE ASSEMBLY_YEAR BETWEEN 2017 AND 2022
        GROUP BY 1 HAVING COUNT(*) >= {min_n}
        ORDER BY LITIGATION_RATE DESC LIMIT {top}
    """)

rate_by("CARRIER_TYPE")

In [ ]:
rate_by("WCIO_NATURE_CODE")  # look codes up in the WCIO nature-of-injury table

In [ ]:
rate_by("WCIO_BODY_CODE")

In [ ]:
rate_by("INDUSTRY_CODE")

In [ ]:
rate_by("DISTRICT_NAME")

In [ ]:
# Reporting lag and age, bucketed
q("""
SELECT
  CASE WHEN REPORT_LAG_DAYS IS NULL THEN 'missing'
       WHEN REPORT_LAG_DAYS <= 7   THEN '0-7 days'
       WHEN REPORT_LAG_DAYS <= 30  THEN '8-30 days'
       WHEN REPORT_LAG_DAYS <= 90  THEN '31-90 days'
       WHEN REPORT_LAG_DAYS <= 365 THEN '91-365 days'
       ELSE '>1 year' END AS REPORT_LAG,
  COUNT(*) AS N, ROUND(AVG(IS_LITIGATED), 3) AS LITIGATION_RATE
FROM WC_ML.ANALYTICS.CLAIM_FEATURES
WHERE ASSEMBLY_YEAR BETWEEN 2017 AND 2022
GROUP BY 1 ORDER BY MIN(COALESCE(REPORT_LAG_DAYS, -1))
""")

In [ ]:
q("""
SELECT FLOOR(AGE_AT_INJURY / 10) * 10 AS AGE_BAND, COUNT(*) AS N, ROUND(AVG(IS_LITIGATED), 3) AS LITIGATION_RATE
FROM WC_ML.ANALYTICS.CLAIM_FEATURES
WHERE ASSEMBLY_YEAR BETWEEN 2017 AND 2022
GROUP BY 1 ORDER BY 1
""")

## 5. Notes to take into modeling

* **Leakage:** the feature view only uses information available when the claim is assembled. Fields such as
  hearing dates, Independent Medical Exam (IME-4) counts, claim status and *Claim Injury Type* are produced
  later, often *because* an attorney got involved. Including them would inflate test scores and fail in production.
* **Report lag** is computed from the WCB *assembly* date. At your own company the comparable intake moment is
  First Notice of Loss (FNOL). Check that the lag definition would be available at scoring time.
* **Class balance:** check the base rate above. It decides whether the precision-recall area under the curve
  (PR-AUC) is a more useful metric than the ROC-AUC (receiver operating characteristic, area under the curve).